# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **Iris Dataset** dari scikit-learn.

Dataset ini berisi 150 sampel bunga iris dengan 4 fitur (sepal length, sepal width, petal length, petal width) dan 3 kelas target (setosa, versicolor, virginica).

Task: **Klasifikasi** - memprediksi spesies bunga iris berdasarkan fitur-fiturnya.

# **2. Import Library**

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# **3. Memuat Dataset**

In [2]:
# Load dataset dari sklearn
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print(f"Shape dataset: {df.shape}")
df.head()

Shape dataset: (150, 6)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,species
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


In [12]:
# Simpan dataset raw
df.to_csv('../iris_raw/iris_raw.csv', index=False)
print("Dataset raw berhasil disimpan.")

Dataset raw berhasil disimpan.


# **4. Exploratory Data Analysis (EDA)**

In [13]:
# Info dataset
print("=== INFO DATASET ===")
print(df.info())
print("\n=== STATISTIK DESKRIPTIF ===")
df.describe()

=== INFO DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
 5   species            150 non-null    object 
dtypes: float64(4), int64(1), object(1)
memory usage: 7.2+ KB
None

=== STATISTIK DESKRIPTIF ===


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


In [14]:
# Cek missing values
print("=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DUPLIKASI ===")
print(f"Jumlah data duplikat: {df.duplicated().sum()}")

print("\n=== DISTRIBUSI TARGET ===")
print(df['species'].value_counts())

=== MISSING VALUES ===
sepal length (cm)    0
sepal width (cm)     0
petal length (cm)    0
petal width (cm)     0
target               0
species              0
dtype: int64

=== DUPLIKASI ===
Jumlah data duplikat: 1

=== DISTRIBUSI TARGET ===
species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


In [15]:
# Cek outlier menggunakan IQR
print("=== DETEKSI OUTLIER (IQR) ===")
numeric_cols = iris.feature_names
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"{col}: {len(outliers)} outlier(s)")

=== DETEKSI OUTLIER (IQR) ===
sepal length (cm): 0 outlier(s)
sepal width (cm): 4 outlier(s)
petal length (cm): 0 outlier(s)
petal width (cm): 0 outlier(s)


# **5. Data Preprocessing**

Tahapan preprocessing:
1. Menghapus data duplikat
2. Menangani outlier dengan IQR method
3. Standarisasi fitur numerik
4. Split data menjadi train dan test set

In [16]:
# 1. Hapus duplikat
df_clean = df.drop(columns=['species']).copy()
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Duplikat dihapus: {before - len(df_clean)} baris")

Duplikat dihapus: 1 baris


In [17]:
# 2. Handling outlier dengan IQR
for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_clean[col] = df_clean[col].clip(lower, upper)

print(f"Shape setelah handling outlier: {df_clean.shape}")

Shape setelah handling outlier: (149, 5)


In [18]:
# 3. Split fitur dan target
X = df_clean.drop(columns=['target'])
y = df_clean['target']

# 4. Standarisasi fitur
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 5. Split train-test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train distribution:\n{y_train.value_counts()}")

X_train shape: (119, 4)
X_test shape: (30, 4)
y_train distribution:
target
0    40
1    40
2    39
Name: count, dtype: int64


In [19]:
# Simpan hasil preprocessing
train_df = X_train.copy()
train_df['target'] = y_train.values
test_df = X_test.copy()
test_df['target'] = y_test.values

train_df.to_csv('iris_preprocessing/iris_train.csv', index=False)
test_df.to_csv('iris_preprocessing/iris_test.csv', index=False)
print("Dataset preprocessing berhasil disimpan.")

Dataset preprocessing berhasil disimpan.
